# 🟡 Interagindo com Neo4j usando Python

Este notebook é um guia didático e prático que demonstra como interagir com o **Neo4j** (banco de dados NoSQL do tipo **Grafos**) utilizando a linguagem Python, o driver oficial `neo4j` e a linguagem de consulta **Cypher**.

## 🛠️ O que é o Neo4j?
O Neo4j armazena dados estruturados como redes de conexões. Em vez de tabelas (SQL) ou documentos (MongoDB), os dados são modelados como:
- **Nós (Nodes):** Entidades (ex: Pessoa, Empresa, Produto).
- **Relacionamentos (Relationships):** Conexões direcionadas com tipos específicos entre nós (ex: `AMIGO_DE`, `TRABALHA_NA`, `COMPROU`).
- **Propriedades (Properties):** Pares chave-valor associados tanto a nós quanto a relacionamentos.

### Detalhes da Conexão Local (Docker Compose):
- **Host:** `localhost`
- **Porta Bolt:** `7687` (protocolo binário de alta performance usado pelo driver)
- **Porta HTTP:** `7474` (painel web do Neo4j Browser)
- **Autenticação:** Nenhuma (`NEO4J_AUTH=none` no docker-compose.yml)

## 1. Instalação do Cliente Python
Para nos conectarmos ao Neo4j, utilizaremos a biblioteca oficial mantida pela própria Neo4j.

In [ ]:
!pip install neo4j

## 2. Conectando ao Banco de Dados
Vamos importar a classe `GraphDatabase` e criar uma instância de driver apontando para a porta Bolt. 

> **Nota:** Como desativamos a autenticação no docker-compose (`NEO4J_AUTH=none`), passamos credenciais vazias ou simplesmente nenhuma credencial de autenticação ativa.

In [ ]:
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"

try:
    # Criar o driver de conexão (sem autenticação no container local)
    driver = GraphDatabase.driver(uri, auth=None)
    
    # Verificar conexão executando uma query rápida
    with driver.session() as session:
        versao = session.run("RETURN dbms.components()[0].versions[0] AS versao").single()["versao"]
        print("✅ Conexão com Neo4j estabelecida com sucesso!")
        print(f"⚡ Versão do Neo4j: {versao}")
except Exception as e:
    print(f"❌ Erro ao conectar ao Neo4j: {e}")
    print("Certifique-se de que o container do Neo4j está rodando e as portas estão mapeadas corretamente.")

## 3. Limpando o Banco de Dados de Testes
Antes de criar novos nós e arestas, vamos rodar um comando Cypher para limpar todos os dados existentes, garantindo que o notebook execute de forma limpa.

In [ ]:
def limpar_banco(tx):
    # DETACH DELETE remove todos os relacionamentos e em seguida os nós correspondentes
    tx.run("MATCH (n) DETACH DELETE n")

with driver.session() as session:
    session.write_transaction(limpar_banco)
    print("🧹 Banco de dados de grafos limpo e pronto!")

## 4. CRUD - Create (Criar Nós e Relacionamentos)
Utilizaremos queries Cypher parametrizadas. A parametrização é recomendada para evitar ataques de injeção e melhorar o cache de execução de planos de query do Neo4j.

In [ ]:
# Função para criar pessoas (nós)
def criar_pessoa(tx, nome, idade, cidade):
    query = """
    CREATE (p:Pessoa {nome: $nome, idade: $idade, cidade: $cidade})
    RETURN p
    """
    tx.run(query, nome=nome, idade=idade, cidade=cidade)

# Função para criar amizades (relacionamentos direcionados com propriedade)
def criar_amizade(tx, nome1, nome2, desde_ano):
    query = """
    MATCH (a:Pessoa {nome: $nome1})
    MATCH (b:Pessoa {nome: $nome2})
    CREATE (a)-[r:AMIGO_DE {desde: $desde_ano}]->(b)
    RETURN r
    """
    tx.run(query, nome1=nome1, nome2=nome2, desde_ano=desde_ano)

# Executar as operações de criação dentro de transações de escrita
with driver.session() as session:
    # 1. Criar os Nós de Pessoa
    session.write_transaction(criar_pessoa, "Alice", 28, "João Pessoa")
    session.write_transaction(criar_pessoa, "Bob", 30, "Recife")
    session.write_transaction(criar_pessoa, "Charlie", 25, "Natal")
    session.write_transaction(criar_pessoa, "Diego", 35, "João Pessoa")
    print("👤 Nós de pessoas criados com sucesso!")
    
    # 2. Criar os Relacionamentos de Amizade (Arestas)
    session.write_transaction(criar_amizade, "Alice", "Bob", 2022)
    session.write_transaction(criar_amizade, "Bob", "Charlie", 2023)
    session.write_transaction(criar_amizade, "Alice", "Diego", 2021)
    print("🔗 Relacionamentos de amizade criados com sucesso!")

## 5. CRUD - Read (Consultar Dados e Caminhos)
Vamos realizar duas consultas:
1. Buscar todos os amigos diretos da Alice.
2. Recomendação de amigos (buscar "amigos de amigos" com quem Alice ainda não é conectada).

In [ ]:
# === Consulta 1: Amigos diretos da Alice ===
query_amigos = """
MATCH (p:Pessoa {nome: $nome_alvo})-[r:AMIGO_DE]->(amigo)
RETURN amigo.nome AS nome, amigo.idade AS idade, r.desde AS ano_amizade
"""

with driver.session() as session:
    resultados = session.run(query_amigos, nome_alvo="Alice")
    print("📖 Amigos diretos da Alice:")
    for registro in resultados:
        print(f"- {registro['nome']} ({registro['idade']} anos), amigos desde {registro['ano_amizade']}")

print("-" * 50)

# === Consulta 2: Recomendação (Amigos de Amigos da Alice) ===
# MATCH encontra o caminho Alice -> Amigo -> AmigoDoAmigo
# O WHERE garante que não estamos recomendando a própria Alice e nem um amigo com quem ela já tem conexão
query_recomendacao = """
MATCH (alice:Pessoa {nome: 'Alice'})-[:AMIGO_DE]->(amigo)-[:AMIGO_DE]->(sugerido)
WHERE NOT (alice)-[:AMIGO_DE]->(sugerido) AND sugerido.nome <> 'Alice'
RETURN sugerido.nome AS recomendacao, amigo.nome AS intermediario
"""

with driver.session() as session:
    recomendacoes = session.run(query_recomendacao)
    print("📊 Recomendações de amizade para Alice:")
    for rec in recomendacoes:
        print(f"- Sugerido: {rec['recomendacao']} (Porque é amigo de {rec['intermediario']})")

## 6. CRUD - Update (Atualizar Nós ou Relacionamentos)
Atualizações no Neo4j utilizam a cláusula `SET` aplicada a nós ou relacionamentos encontrados via `MATCH`.

In [ ]:
# === Atualizar a idade do Bob para 31 anos ===
query_update_idade = """
MATCH (p:Pessoa {nome: $nome})
SET p.idade = $nova_idade
RETURN p.nome AS nome, p.idade AS idade
"""

with driver.session() as session:
    resultado = session.run(query_update_idade, nome="Bob", nova_idade=31).single()
    print(f"🔄 Idade de {resultado['nome']} atualizada com sucesso para {resultado['idade']} anos!")

## 7. CRUD - Delete (Deletar Nós e Relacionamentos)
No Neo4j, você não pode deletar um nó se ele ainda estiver conectado por relacionamentos (isso garante a integridade referencial do grafo). 
Para contornar isso, usamos `DETACH DELETE`, que exclui primeiro as conexões e depois deleta o nó solicitado.

In [ ]:
# === Deletar o nó do Charlie ===
query_delete = """
MATCH (p:Pessoa {nome: $nome_deletar})
DETACH DELETE p
"""

with driver.session() as session:
    session.run(query_delete, nome_deletar="Charlie")
    print("🗑️ Nó de Charlie e todas as suas conexões de amizade foram removidos!")

    # Mostrar pessoas restantes no grafo
    print("\n👥 Pessoas restantes no banco de dados:")
    todos = session.run("MATCH (p:Pessoa) RETURN p.nome AS nome")
    for pessoa in todos:
        print(f"- {pessoa['nome']}")

## 🏁 Conclusão
Parabéns! Você concluiu os testes com o Neo4j e a linguagem Cypher:
- Aprendeu a instanciar e gerenciar conexões via protocolo binário Bolt.
- Criou nós estruturados de forma dinâmica com labels e propriedades.
- Criou relacionamentos tipados entre entidades no grafo.
- Entendeu como navegar por conexões usando padrões em Cypher, incluindo consultas para recomendação (amigo de amigo).
- Atualizou atributos de nós e realizou remoções seguras usando `DETACH DELETE`.

Se quiser ir além, abra a interface web em [http://localhost:7474](http://localhost:7474) para rodar as queries Cypher diretamente e ver a representação visual dos nós e arestas de forma interativa!